# Pool-size-fixed lag-distance sweep (DDM, Colab test)

Deconfounds "how far apart the two compared tokens are" (`k`) from "how many pairs get
mean-pooled" by fixing the pool at 20 pairs, resampled every forward call, at every `k`.

This is a **workflow test**: the same experiment is running locally (CPU, PyTorch) on the full
regime-controlled DDM split (train=1,599,917 / val=399,980). This notebook runs the identical
model/training code on a much smaller random subsample (train=100,000 / val=20,000) so it fits
in a small Drive-hosted file and finishes quickly on Colab's free CPU. Expect the same
qualitative shape (R² rising with `k`) but noisier/somewhat different absolute numbers than the
full local run — 16× fewer training rows.

**Setup:**
1. Upload `ddm_colab_test_split.npz` (~8 MB) to your Google Drive.
2. Run the cells in order. The Drive-mount cell will prompt for authorization.
3. Edit `DATA_PATH` below to match where you placed the file in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH = '/content/drive/MyDrive/ddm_colab_test_split.npz'  # <-- edit if you placed it elsewhere

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# This model is tiny (~2.6K params) and each batch does very little compute, so GPU isn't
# guaranteed to be faster -- kernel-launch and host<->device transfer overhead can dominate
# over the actual FLOPs. Toggle here and compare wall time yourself (per-lag timing is printed
# below); 'cpu' is a reasonable default for this specific experiment.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

npz = np.load(DATA_PATH)
train_tok = npz['train_tokens'].astype(np.int64)
train_target = npz['train_target'].astype(np.float32)
val_tok = npz['val_tokens'].astype(np.int64)
val_target = npz['val_target'].astype(np.float32)
var_val = float(npz['var_val'])

print('train', train_tok.shape, train_target.shape)
print('val  ', val_tok.shape, val_target.shape)
print('Var(val) =', var_val)

OBS_LEN = train_tok.shape[1]
pred_len = train_target.shape[1]

**On GPU vs CPU for this notebook:** if your Colab runtime has a GPU, `device` above picks it up
automatically. Worth comparing empirically — for a model this small (~2.6K params, tiny per-batch
compute), GPU often isn't faster than CPU, since per-batch overhead (kernel launches, host↔device
transfer) tends to dominate over the actual FLOPs. GPU pays off more for the full N² pairwise
variant (much larger per-batch tensors) or a full-size production Transformer, not this pair-MLP.

In [ ]:
LAGS = [1, 2, 4, 8, 16, 32]
POOL_SIZE = 20  # fixed number of pairs pooled at every lag, resampled every forward call
SEEDS = [1, 2, 3]
N_EPOCHS = 15
BATCH_SIZE = 256
LR = 1e-3

for k in LAGS:
    assert OBS_LEN - k >= POOL_SIZE, f'lag {k} leaves only {OBS_LEN-k} positions, need >= {POOL_SIZE}'

In [ ]:
class PoolFixedLagModel(nn.Module):
    # z_i(k) = phi([x_i, x_{i+k}]) -- phi sees the two raw (quantize_diff bin index) token
    # values directly, no embedding layer, so any distance-k structure found can't be
    # attributed to embedding-space geometry the model was free to invent. Pools over a
    # fixed-size random subset of valid (i, i+k) positions -- POOL_SIZE of them, resampled
    # every forward call -- so the number of pairs averaged is identical at every lag,
    # deconfounding "distance k" from "how many pairs got pooled."
    def __init__(self, hidden, pred_len, lag):
        super().__init__()
        self.lag = lag
        self.phi = nn.Sequential(
            nn.Linear(2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.rho = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, pred_len))

    def forward(self, tokens):
        x = tokens.float()
        n = x.shape[1]
        k = self.lag
        n_valid = n - k
        positions = np.random.choice(n_valid, size=POOL_SIZE, replace=False)
        positions = torch.tensor(positions, dtype=torch.long)
        xi = x[:, positions]
        xj = x[:, positions + k]
        pairs = torch.stack([xi, xj], dim=-1)  # [B, POOL_SIZE, 2]
        h = self.phi(pairs)
        pooled = h.mean(dim=1)
        return self.rho(pooled)

In [ ]:
def train_lag(lag, seed, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = PoolFixedLagModel(32, pred_len, lag).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()

    tr_x = torch.tensor(train_tok, dtype=torch.long, device=device)
    tr_y = torch.tensor(train_target, dtype=torch.float32, device=device)
    va_x = torch.tensor(val_tok, dtype=torch.long, device=device)
    va_y = torch.tensor(val_target, dtype=torch.float32, device=device)

    n = len(tr_x)
    best_val = float('inf')
    for epoch in range(n_epochs):
        model.train()
        order = np.random.permutation(n)
        for i in range(0, n - batch_size, batch_size):
            idx = order[i:i + batch_size]
            opt.zero_grad()
            loss = crit(model(tr_x[idx]), tr_y[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        losses = []
        with torch.no_grad():
            for i in range(0, len(va_x) - batch_size, batch_size):
                losses.append(crit(model(va_x[i:i + batch_size]), va_y[i:i + batch_size]).item())
        val_mse = float(np.mean(losses))
        best_val = min(best_val, val_mse)
    return best_val

In [ ]:
import time

results = {}
for lag in LAGS:
    t0 = time.time()
    seed_r2 = []
    for seed in SEEDS:
        mse = train_lag(lag, seed)
        r2 = 1 - mse / var_val
        seed_r2.append(r2)
        print(f'lag={lag:3d} seed={seed}: mse={mse:.6f} r2={r2:.4f}')
    results[lag] = {'r2_seeds': seed_r2, 'r2_mean': float(np.mean(seed_r2))}
    print(f'  -> lag={lag} mean R2={results[lag]["r2_mean"]:.4f}  ({time.time()-t0:.1f}s)')

In [ ]:
print('=== SUMMARY (pool size fixed at 20) ===')
for lag in LAGS:
    r = results[lag]
    print(f'k={lag:3d}  R2={r["r2_mean"]:.4f}  seeds={[round(x,4) for x in r["r2_seeds"]]}')

In [ ]:
import matplotlib.pyplot as plt

ks = list(results.keys())
means = [results[k]['r2_mean'] for k in ks]
plt.figure(figsize=(6, 4))
plt.plot(ks, means, marker='o')
plt.xscale('log', base=2)
plt.xlabel('lag k (log scale)')
plt.ylabel('R²')
plt.title('Pool-size-fixed lag sweep (DDM subsample, Colab)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import json

out_path = '/content/drive/MyDrive/pool_fixed_lag_sweep_colab_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print('saved to', out_path)